# NHS RTT breach risk classification

I build a monthly panel of provider level breach rates by engineering a small set of
features from prior months only. I then trained a logistic regression to predict
whether a provider will perform worse than the national average in a given month.

With only 6 months of data I do not have enough history for a real time series
forecast, so I am not attempting one here. What I do have is close to 3,000
provider month rows, a real sample size for a classification task, so that is
what this notebook focuses on. I split train and test by time, training on
Dec 2025 to Mar 2026 and testing on Apr 2026, so the model only ever sees the
future during evaluation, never during training.

Setup And Imports:

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

DB_USER = "root"
DB_PASSWORD = "optima123!"
DB_HOST = "127.0.0.1"
DB_PORT = 3306
DB_NAME = "nhs_rtt_analytics"

engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## Step 1: pull the provider month panel

One row per provider per month, with total patients, breach rate, and region.
I join through to region here since the EDA showed it genuinely relates to
breach rate.

In [2]:
panel = pd.read_sql('''
    SELECT
        p.provider_code,
        p.provider_name,
        f.period_date,
        r.region_name,
        SUM(f.patient_count) AS total_patients,
        SUM(CASE WHEN b.breach_flag = TRUE THEN f.patient_count ELSE 0 END) * 100.0
            / SUM(f.patient_count) AS breach_rate_pct
    FROM fact_rtt_waiting_times f
    JOIN dim_providers p ON f.provider_code = p.provider_code
    JOIN dim_weeks_bands b ON f.band_id = b.band_id
    LEFT JOIN dim_icb_region_map m ON p.provider_parent_code = m.icb_code
    LEFT JOIN dim_regions r ON m.region_code = r.region_code
    GROUP BY p.provider_code, p.provider_name, f.period_date, r.region_name
    HAVING SUM(f.patient_count) >= 100
''', engine)

panel["period_date"] = pd.to_datetime(panel["period_date"])
panel = panel.sort_values(["provider_code", "period_date"]).reset_index(drop=True)
panel = panel.dropna(subset=["region_name"])

print(f"{len(panel):,} provider month rows, {panel['provider_code'].nunique()} providers")
panel.head(10)

2,864 provider month rows, 497 providers


,provider_code,provider_name,period_date,region_name,total_patients,breach_rate_pct
0,A0C5S,SPAMEDICA GLOUCESTER,2025-11-01,South West,182.0,8.79121
1,A0C5S,SPAMEDICA GLOUCESTER,2025-12-01,South West,224.0,10.26786
2,A0C5S,SPAMEDICA GLOUCESTER,2026-01-01,South West,164.0,9.75610
3,A0C5S,SPAMEDICA GLOUCESTER,2026-02-01,South West,206.0,12.13592
4,A0C5S,SPAMEDICA GLOUCESTER,2026-03-01,South West,184.0,9.23913
5,A0C5S,SPAMEDICA GLOUCESTER,2026-04-01,South West,223.0,8.96861
6,A1D1B,SPAMEDICA WEMBLEY,2025-11-01,London,547.0,3.29068
7,A1D1B,SPAMEDICA WEMBLEY,2025-12-01,London,544.0,3.86029
8,A1D1B,SPAMEDICA WEMBLEY,2026-01-01,London,609.0,4.59770
9,A1D1B,SPAMEDICA WEMBLEY,2026-02-01,London,746.0,4.28954


## Step 2: feature engineering

Every feature here is calculated only from months strictly before the row's
own month, so the model never sees information it would not actually have
at prediction time. The target is relative to that month's national average
rather than a fixed percentage, since the national average itself moved from
39.34% to 36.79% across these 6 months.

In [3]:
panel["prior_month_breach_rate_pct"] = panel.groupby("provider_code")["breach_rate_pct"].shift(1)

panel["avg_breach_rate_to_date"] = (
    panel.groupby("provider_code")["breach_rate_pct"]
    .apply(lambda s: s.shift(1).expanding().mean())
    .reset_index(level=0, drop=True)
)

panel["log_total_patients"] = np.log(panel["total_patients"])

national_avg_by_month = panel.groupby("period_date")["breach_rate_pct"].transform("mean")
panel["is_high_risk"] = (panel["breach_rate_pct"] > national_avg_by_month).astype(int)

model_data = panel.dropna(subset=["prior_month_breach_rate_pct", "avg_breach_rate_to_date"]).copy()
print(f"{len(model_data):,} rows usable after dropping each provider's first month")
model_data[["provider_name", "period_date", "breach_rate_pct", "prior_month_breach_rate_pct", "avg_breach_rate_to_date", "is_high_risk"]].head(10)

2,367 rows usable after dropping each provider's first month


,provider_name,period_date,breach_rate_pct,prior_month_breach_rate_pct,avg_breach_rate_to_date,is_high_risk
1,SPAMEDICA GLOUCESTER,2025-12-01,10.26786,8.79121,8.791210,0
2,SPAMEDICA GLOUCESTER,2026-01-01,9.75610,10.26786,9.529535,0
3,SPAMEDICA GLOUCESTER,2026-02-01,12.13592,9.75610,9.605057,0
4,SPAMEDICA GLOUCESTER,2026-03-01,9.23913,12.13592,10.237773,0
5,SPAMEDICA GLOUCESTER,2026-04-01,8.96861,9.23913,10.038044,0
7,SPAMEDICA WEMBLEY,2025-12-01,3.86029,3.29068,3.290680,0
8,SPAMEDICA WEMBLEY,2026-01-01,4.59770,3.86029,3.575485,0
9,SPAMEDICA WEMBLEY,2026-02-01,4.28954,4.59770,3.916223,0
10,SPAMEDICA WEMBLEY,2026-03-01,7.36278,4.28954,4.009552,0
11,SPAMEDICA WEMBLEY,2026-04-01,9.50276,7.36278,4.680198,0


## Step 3: train and test split by time

Training on Dec 2025 to Mar 2026, testing on Apr 2026, my true holdout month.

In [4]:
train = model_data[model_data["period_date"] < "2026-04-01"]
test = model_data[model_data["period_date"] == "2026-04-01"]
print(f"Train rows: {len(train):,}  ({train['period_date'].min().date()} to {train['period_date'].max().date()})")
print(f"Test rows:  {len(test):,}  (2026-04-01, held out entirely from training)")

feature_cols_numeric = ["prior_month_breach_rate_pct", "avg_breach_rate_to_date", "log_total_patients"]
region_dummies_train = pd.get_dummies(train["region_name"], prefix="region")
region_dummies_test = pd.get_dummies(test["region_name"], prefix="region")
region_dummies_test = region_dummies_test.reindex(columns=region_dummies_train.columns, fill_value=0)

X_train = pd.concat([train[feature_cols_numeric].reset_index(drop=True), region_dummies_train.reset_index(drop=True)], axis=1)
X_test = pd.concat([test[feature_cols_numeric].reset_index(drop=True), region_dummies_test.reset_index(drop=True)], axis=1)
y_train = train["is_high_risk"].reset_index(drop=True)
y_test = test["is_high_risk"].reset_index(drop=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train rows: 1,876  (2025-12-01 to 2026-03-01)
Test rows:  491  (2026-04-01, held out entirely from training)


## Step 4: train and evaluate

In [5]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print(f"Accuracy:  {accuracy_score(y_test, predictions):.3f}")
print(f"Precision: {precision_score(y_test, predictions):.3f}")
print(f"Recall:    {recall_score(y_test, predictions):.3f}")
print("\nConfusion matrix (rows are actual, columns are predicted):")
print(confusion_matrix(y_test, predictions))
print(f"\n{classification_report(y_test, predictions, target_names=['Not high risk', 'High risk'])}")

Accuracy:  0.949
Precision: 0.939
Recall:    0.970

Confusion matrix (rows are actual, columns are predicted):
[[204  17]
 [  8 262]]

               precision    recall  f1-score   support

Not high risk       0.96      0.92      0.94       221
    High risk       0.94      0.97      0.95       270

     accuracy                           0.95       491
    macro avg       0.95      0.95      0.95       491
 weighted avg       0.95      0.95      0.95       491



## Step 5: explainability

Logistic regression coefficients show how much each feature pushes a
prediction toward or away from high risk.

In [6]:
coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

coefficients

,feature,coefficient
0,prior_month_breach_rate_pct,4.340207
1,avg_breach_rate_to_date,1.107553
2,log_total_patients,0.410320
8,region_South East,-0.135483
7,region_North West,0.115723
4,region_London,-0.115389
5,region_Midlands,0.093667
3,region_East of England,0.049954
6,region_North East and Yorkshire,-0.038254
9,region_South West,0.024390
